In [1]:
# import libraries
import pandas as pd
import ee
import geemap

## Connect to Google Earth Engine (GEE)

In [ ]:
# Authenticate GEE
ee.Authenticate()

# Initialize GEE
EE_PROJECT_ID = ""   # Change to your project ID

#ee.Initialize(project=EE_PROJECT_ID)
ee.Initialize()

## . Visualization Parameters

In [4]:
# Boundary visualization params 
vis_params_fao_1 = {
  "fillColor": 'b5ffb4',
  "color": '00909F',
  "width": 1.0,
}

vis_params_aoi = {"fillcolor": "", "color": "red"}

## Boundary Data

In [5]:
# Boundary data from FAO GAUL
# Data source: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_SIMPLIFIED_500m_2015_level0

fao_gaul_l0 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0') # Countries boundaries
fao_gaul_l1 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # States boundaries
fao_gaul_l2 = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level1') # LGAs boundaries



# Creat map to visualize the FAO GAUL boundary data
boundary_map = geemap.Map(center=(7.0, 8.0), zoom=10)

boundary_map.addLayer(fao_gaul_l0, {}, 'Country Boundaries')
boundary_map.addLayer(fao_gaul_l1, {}, 'State Boundaries')
boundary_map.addLayer(fao_gaul_l2, {}, 'LGA Boundaries')
boundary_map

Map(center=[7.0, 8.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [6]:
# Check columns
print(fao_gaul_l0.limit(0).getInfo()["columns"])

# Extract Nigerian boundaries from FAO GAUL
nga_l0 = fao_gaul_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
nga_l1 = fao_gaul_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria"))
fct_l0 = nga_l1.filter(ee.Filter.eq("ADM1_NAME", "Abuja")) # Abuja/FCT extent
print(fct_l0.getInfo())

# Get geometry from Abuja boundary (FeatureCollection)
aoi = fct_l0.geometry()
aoi_bbox = aoi.bounds()

# Creat map to visualize AbujaS boundary data
aoi_map = geemap.Map(center=(7.0, 8.0), zoom=10)
aoi_map.addLayer(fct_l0, vis_params_aoi, 'Abuja Boundary')
aoi_map

{'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'DISP_AREA': 'String', 'EXP0_YEAR': 'Integer', 'STATUS': 'String', 'STR0_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'system:index': 'String'}
{'type': 'FeatureCollection', 'columns': {'ADM0_CODE': 'Integer', 'ADM0_NAME': 'String', 'ADM1_CODE': 'Integer', 'ADM1_NAME': 'String', 'DISP_AREA': 'String', 'EXP1_YEAR': 'Integer', 'STATUS': 'String', 'STR1_YEAR': 'Integer', 'Shape_Area': 'Float', 'Shape_Leng': 'Float', 'system:index': 'String'}, 'version': 1701682755394127, 'id': 'FAO/GAUL_SIMPLIFIED_500m/2015/level1', 'properties': {'system:asset_size': 80042928}, 'features': [{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[6.746250262158974, 8.996497434811962], [6.748495969531178, 8.992005993451773], [6.748495969531178, 8.989760187648406], [6.748495969531178, 8.985268685352079], [6.748495969531178, 8.983022905093227], [6.748495969531178, 8.98077718923787], [6.750741754879445, 8.971794157254672], [6.757

Map(center=[7.0, 8.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

## Explore Image operations

In [29]:
# Image Collection
s2_img_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') # All Sentinel-2 collection
    .filterDate('2022-01-01', '2022-01-31') # Limit/filter to January 2022
    .filterBounds(fct_l0.geometry()) # Limit/filter to Abuja
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# Collection Properties
print(f"Number of images in S2 collection: {s2_img_col.size().getInfo()}\n")

# First image in collection
first_s2_img = s2_img_col.first()
print(f"First image in S2 collection: {first_s2_img.getInfo()}\n")

# Bands in S2 image
print(first_s2_img.bandNames().getInfo())

# Select just "B4"
red_band_s2 = first_s2_img.select(["B4", "B3", "B2"])
print(f"Red Band: {red_band_s2.bandNames().getInfo()}")


# Visualize S2 image
vis_params_s2 = {"min" : 300, "max" :3000, "bands": ["B8", "B4", "B3"]}
s2_map = geemap.Map(center = [7.08, 8.88], zoom=8)
s2_map.addLayer(first_s2_img, vis_params_s2, "First S2 Img")
s2_map


Number of images in S2 collection: 20

First image in S2 collection: {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [1830, 1830], 'crs': 'EPSG:32632', 'crs_transform': [60, 0, 199980, 0, -60, 1000020]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32632', 'crs_transform': [10, 0, 199980, 0, -10, 1000020]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32632', 'crs_transform': [10, 0, 199980, 0, -10, 1000020]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32632', 'crs_transform': [10, 0, 199980, 0, -10, 1000020]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [5490, 5490], 

Map(center=[7.08, 8.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [27]:
# Mosaic
s2_img_median = s2_img_col.mean()
print(f"Median of S2 collection: {s2_img_median.getInfo()}")

Median of S2 collection: {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B6', 'data_type': {'type': 'PixelType', 'precision': 'double', 'min': 0, 'max': 65535}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'B7', 'data_type

## . Land Cover Data Download

### . Land Cover / Existing Urban Layer (GLC_FCS30D: 2012–2022)

The raw GLC_FCS30D asset is stored as tiled, multi-band images (one band per
year). We mosaic the tiles, rename the bands to actual years, and convert the
multi-band image into a proper year-indexed `ImageCollection`, following the
official preprocessing pattern published by the data provider.

In the GLC_FCS30D legend, class code **190 = Impervious surfaces** (built-up /
urban). We use this to derive a binary urban mask for every year.


In [ ]:
# GLC_FCS30D Global Land Cover
# Data source : https://gee-community-catalog.org/projects/glc_fcs/?h=glc+fcs30d
# Reference   : https://gee-community-catalog.org/tutorials/examples/glc_fcs30d_lulc/

# Annual land cover ImageCollection (2000 – 2022). 
# Each image in annual land cover data has 23 bands, one for each year from 2000-2022 (23 years)
# (bands b1, b2,..b23 = 2000, 2001,..2022)
glc_annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")

# Classification scheme
# (35 landcover class and 1 fill value)
glc_class_values =  [
  10, 11, 12, 20, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 120, 121, 122, 
  130, 140, 150, 152, 153, 181, 182, 183, 184, 185, 186, 187, 190, 200, 
  201, 202, 210, 220, 0
]

# Land cover class names
glc_class_names = [
    "Rainfed_cropland", "Herbaceous_cover_cropland", "Tree_or_shrub_cover_cropland",
    "Irrigated_cropland", "Open_evergreen_broadleaved_forest", "Closed_evergreen_broadleaved_forest",
    "Open_deciduous_broadleaved_forest", "Closed_deciduous_broadleaved_forest",
    "Open_evergreen_needle_leaved_forest", "Closed_evergreen_needle_leaved_forest",
    "Open_deciduous_needle_leaved_forest", "Closed_deciduous_needle_leaved_forest",
    "Open_mixed_leaf_forest", "Closed_mixed_leaf_forest", "Shrubland",
    "Evergreen_shrubland", "Deciduous_shrubland", "Grassland", "Lichens_and_mosses",
    "Sparse_vegetation", "Sparse_shrubland", "Sparse_herbaceous", "Swamp", "Marsh",
    "Flooded_flat", "Saline", "Mangrove", "Salt_marsh", "Tidal_flat",
    "Impervious_surfaces", "Bare_areas", "Consolidated_bare_areas",
    "Unconsolidated_bare_areas", "Water_body", "Permanent_ice_and_snow", "Filled_value",
]

glc_class_colours = [
    "#ffff64", "#ffff64", "#ffff00", "#aaf0f0", "#4c7300",
    "#006400", "#a8c800", "#00a000", "#005000", "#003c00",
    "#286400", "#285000", "#a0b432", "#788200", "#966400",
    "#964b00", "#966400", "#ffb432", "#ffdcd2", "#ffebaf",
    "#ffd278", "#ffebaf", "#00a884", "#73ffdf", "#9ebb3b",
    "#828282", "#f57ab6", "#66cdab", "#444f89", "#c31400",
    "#fff5d7", "#dcdcdc", "#fff5d7", "#0046c8", "#ffffff", "#ffffff",
]

In [ ]:
# Mosaic tiled images and rename bands b1, b2, ... to 2000, 2001, ...
glc_mosaic = glc_annual.mosaic()
years_list = ee.List.sequence(2000, 2022).map(lambda year: ee.Number(year).format("%04d"))
glc_mosaic_renamed = glc_mosaic.rename(years_list)

# Multiband to single-band annual mosaic & assign time and year metadata to each 
glc_mosaic_renamed_upd = years_list.map(
    lambda year: glc_mosaic_renamed
    .select([year])
    .set({
        "system:time_start": ee.Date.fromYMD(ee.Number.parse(year), 1, 1).millis(),
        "system:index": year,
        "year": ee.Number.parse(year),
    })
)

glc_mosaics_col = ee.ImageCollection.fromImages(glc_mosaic_renamed_upd)
print(glc_mosaics_col.first().getInfo())

# Remap native class values to sequential integers
new_class_values = ee.List.sequence(1, ee.List(glc_class_values).length())

## Drivers of Urban Growth / Predictors

**Elevation/DEM**

In [39]:
# Download elevation and compute slope
dem = ee.Image('USGS/SRTMGL1_003')
dem.bandNames().getInfo()



elevation = dem.select('elevation')
slope = ee.Terrain.slope(elevation)


dem_map = geemap.Map(center = [7.08, 8.88], zoom=8)
dem_map.add_basemap("SATELLITE")
dem_map.addLayer(dem.clip(aoi), {'min': 0, 'max': 500, "palette": ["Red", "Green", "Yellow"]}, "DEM")
dem_map.addLayer(slope.clip(aoi), {'min': 0, 'max': 10, "palette": ["Red", "Green", "Yellow"]}, "Slope")
dem_map


Map(center=[7.08, 8.88], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [ ]:
""" 



m = geemap.Map()
m.set_center(-112.8598, 36.2841, 10)
m.add_layer(slope, {'min': 0, 'max': 60}, 'slope')
m
"""